# Data Preperation

Import libraries 

In [762]:
import os
import numpy as np
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split 


Loading the dataset

In [ ]:
# Defining the directory containing the dataset
datasets = str("C:/../GitHub/Google-Image-Scraper/photos")

# Folder paths to the images of each class are stored
folder_paths = [
    r"C:/../GitHub/Google-Image-Scraper/photos/Cactus",
    r"C:/../GitHub/Google-Image-Scraper/photos/Calathea",
    r"C:/../GitHub/Google-Image-Scraper/photos/Monstera Minima",
    r"C:/../GitHub/Google-Image-Scraper/photos/Peperomia",
    r"C:/../GitHub/Google-Image-Scraper/photos/Pilea Peperomioides",
    r"C:/../GitHub/Google-Image-Scraper/photos/Pothos"
]

# Class labels for the images (same order as folder_paths)
class_names = [
    "Cactus", "Calathea", "Monstera Minima", "Peperomia", "Pilea Peperomioides", "Pothos"
]

I will implement LabelEncoder() for the classes. This is for the reason that, the Machine Learning algorithm works with numerical data. By using LabelEncoder(), I can convert the folder names into numerical labels, which can then be used as input features for the model.

In [764]:
# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform folder names
encoded_classes = label_encoder.fit_transform(class_names)

# Print encoded categories
print(encoded_classes)

[0 1 2 3 4 5]


Ensuring that every image in the dataset is in 'RGB' format and each class contains approximately 200 images. This is for the reason that, during the creation of the dataset, using Google Image Scraper, manual verification was necessary for each photo due to inconsistencies in image quality. For example, some pictures contained multiple plants with varying plant species, this would complicate the identifying process. Consequently, there are variations in the number of images across classes. Achieving an equal distribution of images among all classes would be preferable.

In [765]:
from skimage.color import rgba2rgb
from skimage import io, transform, color

def is_rgb_image(image_path):
    try:
        with Image.open(image_path) as img:
            if img.mode == 'RGB':
                return True
            elif img.mode == 'RGBA':
                # Convert RGBA to RGB
                img = img.convert("RGB")
                return True
            else:
                return False
    except Exception as e:
        print(f"Error: {e}")
        return False
    
# Initializing dictionary to store images for each class
cleaned_data = {encoded_class: [] for encoded_class in encoded_classes}

# Maximum number of images per class
max_images_per_class = 200

# Iterate through each folder
for folder_path, encoded_class in zip(folder_paths, encoded_classes):
    # Count the number of images for the class
    count = 0
    # Iterate through each file in the folder
    for filename in os.listdir(folder_path):
        # Check if the file is an image and in RGB format
        image_path = os.path.join(folder_path, filename)
        if is_rgb_image(image_path):
            # Add the image to the dataset for the class
            cleaned_data[encoded_class].append(image_path)
            count += 1
        # Break if the maximum number of images per class is reached
        if count >= max_images_per_class:
            break

# Now let's check if all images in each class are RGB
for encoded_class, images in cleaned_data.items():
    all_rgb = all(is_rgb_image(image_path) for image_path in images)
    print(f"Class: {encoded_class}, Number of Images: {len(images)}, All RGB: {all_rgb}")

Class: 0, Number of Images: 200, All RGB: True
Class: 1, Number of Images: 200, All RGB: True
Class: 2, Number of Images: 200, All RGB: True
Class: 3, Number of Images: 200, All RGB: True
Class: 4, Number of Images: 200, All RGB: True
Class: 5, Number of Images: 200, All RGB: True


The code below is designed to divide the dataset into three subsets: a training set comprising 70% of the data, a validation set containing 20%, and a test set consisting of 10%. I performed manual calculations, to ensure the accuracy of these sets.

Given that there are 6 classes, each containing 200 images, the total dataset comprises 1200 images. Consequently, the training set should encompass 840 images, the validation set should include 240 images, and the test set should contain 120 images.

In [766]:
# Combine all images and labels into a single list
X = []
y = []
for encoded_class, images in cleaned_data.items():
    X.extend(images)
    y.extend([encoded_class] * len(images))

# Split the data into train, validation, and test sets (70% train, 20% validation, 10% test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=True) # X_temp and y_temp created to store the remaining 30% of the dataset
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.333, random_state=42, shuffle=True) # X_temp and y_temp used to divide the remaining 30% of the dataset

# Check the sizes of each set
print("Train set size:", len(X_train))
print("Validation set size:", len(X_val))
print("Test set size:", len(X_test))

Train set size: 840
Validation set size: 240
Test set size: 120


I encoutnered several errors during the iteration process from task 3.2 to 3.6, which led me to discover that x_train, x_val and X_test contained strings instead of arrays. To address this issue, it was crucial to convert these variables to NumPy arrays. By converting them, it ensures that the data is appropriately formatted for the Machine Learning task, thereby reducing errors and enhancing the overall reliability of the process.

In [767]:
import cv2
# implementing a defintion to resize, normalize and ensure the image are in rgb format.
def load_image(image_path, target_size=(128, 128)):
    # Read image in BGR format
    img = cv2.imread(image_path)
    # Resize image
    img = cv2.resize(img, target_size)
    # Convert color channels from BGR to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(float) / 255.0
    return img

In [768]:
# Load images and convert them to arrays
X_train = [load_image(images) for images in X_train]
X_val = [load_image(images) for images in X_val]
X_test = [load_image(images) for images in X_test]

# Convert y_train, y_val, and y_test to numpy arrays
import numpy as np
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

# Convert X_train, X_val, and X_test to numpy arrays
X_train = np.array(X_train)
X_val = np.array(X_val)
X_test = np.array(X_test)

# Check the shape of the first image, X
print("X_train shape of the first image:", X_train[0].shape)
print("X_val shape of the first image:", X_val[0].shape)
print("X_test shape of the first image:", X_test[0].shape)


X_train shape of the first image: (128, 128, 3)
X_val shape of the first image: (128, 128, 3)
X_test shape of the first image: (128, 128, 3)
